# Feature Engineering

Purpose:<br>
Create predictors available at or before quarter t for predicting whether a
MySuper investment option will fall into the bottom quartile of peer performance
over quarters t+1 to t+4.

Target:<br>
future_4q_bottom_quartile

Rule:<br>
No feature may use information after period_end_date.

## Data Import from audit file

In [1]:
import pandas as pd
from pathlib import Path

processed_dir = Path("../data/processed")

hist_model_mysuper_features = pd.read_parquet(
    processed_dir / "hist_model_mysuper_target_peer.parquet"
)

In [2]:
hist_model_mysuper_features.shape

(13719, 110)

In [3]:
hist_model_mysuper_features[
    "future_4q_target_available_option"
].value_counts(dropna=False)

future_4q_target_available_option
True     11720
False     1999
Name: count, dtype: int64

In [4]:
hist_model_mysuper_features[
    [
        "period_end_date",
        "target_end_date",
        "future_4q_bottom_quartile",
    ]
].dtypes

period_end_date              datetime64[us]
target_end_date              datetime64[us]
future_4q_bottom_quartile           boolean
dtype: object

In [5]:
# define entity keys
entity_keys = [
    "rse_abn",
    "abn_product_identifier",
    "abn_investment_menu_identifier",
    "abn_investment_option_identifier",
]

# sort values based on entity keys and quarter
hist_model_mysuper_features = (
    hist_model_mysuper_features
    .sort_values(
        entity_keys + ["period_end_date"]
    )
    .reset_index(drop=True)
)

## Setting performance feature set

In [6]:
return_col = "return_measurement_comparison_percent"

# create lagged return columns
grouped_returns = (
    hist_model_mysuper_features
    .groupby(entity_keys)[return_col]
)

hist_model_mysuper_features["return_lag_1q"] = (
    grouped_returns.shift(1)
)

hist_model_mysuper_features["return_lag_2q"] = (
    grouped_returns.shift(2)
)

hist_model_mysuper_features["return_lag_4q"] = (
    grouped_returns.shift(4)
)

interpretation:<br>
- return_measurement_comparison_percent = return at t
- return_lag_1q                         = return at t-1
- return_lag_2q                         = return at t-2
- return_lag_4q                         = return at t-4

In [7]:
# add deterioration feature
# using an early-warning concept to know whether performance has weakened relative to the previous quarter
hist_model_mysuper_features[
    "return_change_1q"
] = (
    hist_model_mysuper_features[
        return_col
    ]
    -
    hist_model_mysuper_features[
        "return_lag_1q"
    ]
)

In [8]:
# create quater identifiers and lagged quarter dates
# validate whether the lag created before really 1,2,3,4 quarter ago
hist_model_mysuper_features[
    "quarter"
] = (
    hist_model_mysuper_features[
        "period_end_date"
    ].dt.to_period("Q")
)

grouped_quarters = (
    hist_model_mysuper_features
    .groupby(entity_keys)["quarter"]
)

hist_model_mysuper_features[
    "quarter_lag_1q"
] = grouped_quarters.shift(1)

hist_model_mysuper_features[
    "quarter_lag_2q"
] = grouped_quarters.shift(2)

hist_model_mysuper_features[
    "quarter_lag_4q"
] = grouped_quarters.shift(4)

# Check whether each lag is genuinely the requested quarter
hist_model_mysuper_features[
    "return_lag_1q_valid"
] = (
    hist_model_mysuper_features["quarter_lag_1q"]
    ==
    hist_model_mysuper_features["quarter"] - 1
)

hist_model_mysuper_features[
    "return_lag_2q_valid"
] = (
    hist_model_mysuper_features["quarter_lag_2q"]
    ==
    hist_model_mysuper_features["quarter"] - 2
)

hist_model_mysuper_features[
    "return_lag_4q_valid"
] = (
    hist_model_mysuper_features["quarter_lag_4q"]
    ==
    hist_model_mysuper_features["quarter"] - 4
)

In [9]:
# inspect result
hist_model_mysuper_features[
    [
        "return_lag_1q_valid",
        "return_lag_2q_valid",
        "return_lag_4q_valid",
    ]
].apply(
    lambda x: x.value_counts(dropna=False)
)

,return_lag_1q_valid,return_lag_2q_valid,return_lag_4q_valid
True,13204,12689,11674
False,515,1030,2045


this pattern looks largely like the beginning of each investment option's history. For example, the first observation of an option cannot have a 1-quarter lag; the first two cannot have a 2-quarter lag; the first four cannot have a 4-quarter lag.

But False might have 2 meanings:
- No earlier observation exists → expected insufficient history
- An earlier observation exists, but it is not the required calendar quarter → genuine gap

The second meaning has to be handled

In [10]:
# check actual gap
for lag in [1, 2, 4]:
    valid_col = f"return_lag_{lag}q_valid"
    quarter_lag_col = f"quarter_lag_{lag}q"

    actual_gap = (
        ~hist_model_mysuper_features[valid_col]
        &
        hist_model_mysuper_features[quarter_lag_col].notna()
    )

    no_prior_history = (
        ~hist_model_mysuper_features[valid_col]
        &
        hist_model_mysuper_features[quarter_lag_col].isna()
    )

    print(f"{lag}Q lag")
    print("Valid:", hist_model_mysuper_features[valid_col].sum())
    print("No prior history:", no_prior_history.sum())
    print("Actual gap:", actual_gap.sum())
    print()

1Q lag
Valid: 13204
No prior history: 515
Actual gap: 0

2Q lag
Valid: 12689
No prior history: 1030
Actual gap: 0

4Q lag
Valid: 11674
No prior history: 2045
Actual gap: 0



In [11]:
import numpy as np

# mask the invalid lag values
for lag in [1, 2, 4]:
    hist_model_mysuper_features.loc[
        ~hist_model_mysuper_features[
            f"return_lag_{lag}q_valid"
        ],
        f"return_lag_{lag}q",
    ] = np.nan

# recalculate deterioration
hist_model_mysuper_features[
    "return_change_1q"
] = (
    hist_model_mysuper_features[
        "return_measurement_comparison_percent"
    ]
    -
    hist_model_mysuper_features[
        "return_lag_1q"
    ]
)

#### Create trailing 4-quarter cumulative return

In [12]:
# create 3-quarter lag
hist_model_mysuper_features[
    "return_lag_3q"
] = (
    hist_model_mysuper_features
    .groupby(entity_keys)[
        "return_measurement_comparison_percent"
    ]
    .shift(3)
)

# Create trailing cumulative return
hist_model_mysuper_features[
    "return_trailing_4q"
] = (
    (1 + hist_model_mysuper_features[
        "return_measurement_comparison_percent"
    ])
    *
    (1 + hist_model_mysuper_features[
        "return_lag_1q"
    ])
    *
    (1 + hist_model_mysuper_features[
        "return_lag_2q"
    ])
    *
    (1 + hist_model_mysuper_features[
        "return_lag_3q"
    ])
    - 1
)

In [13]:
# check distribution
hist_model_mysuper_features[
    "return_trailing_4q"
].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
)

count    12176.000000
mean         0.075215
std          0.071414
min         -0.114451
1%          -0.084385
5%          -0.058948
25%          0.036310
50%          0.082569
75%          0.115614
95%          0.191143
99%          0.255577
max          0.372972
Name: return_trailing_4q, dtype: float64

In [14]:
hist_model_mysuper_features[
    "return_trailing_4q"
].isna().value_counts()

return_trailing_4q
False    12176
True      1543
Name: count, dtype: int64

In [15]:
return_col = "return_measurement_comparison_percent"

# group the table based on previous entity keys
grouped_returns = (
    hist_model_mysuper_features
    .groupby(entity_keys)[return_col]
)

# calculate mean over 4 quarters
hist_model_mysuper_features[
    "return_mean_4q"
] = (
    grouped_returns
    .rolling(window=4, min_periods=4)
    .mean()
    .reset_index(level=entity_keys, drop=True)
)

# calculate standard deviation over 4 quarters
hist_model_mysuper_features[
    "return_std_4q"
] = (
    grouped_returns
    .rolling(window=4, min_periods=4)
    .std()
    .reset_index(level=entity_keys, drop=True)
)

# calculate minimum/worst return during 4 quarters
hist_model_mysuper_features[
    "return_min_4q"
] = (
    grouped_returns
    .rolling(window=4, min_periods=4)
    .min()
    .reset_index(level=entity_keys, drop=True)
)

# calculate positive proportion of the return over 4 quarters
hist_model_mysuper_features[
    "positive_quarter_share_4q"
] = (
    grouped_returns
    .rolling(window=4, min_periods=4)
    .apply(lambda x: (x > 0).mean(), raw=True)
    .reset_index(level=entity_keys, drop=True)
)

<b>Meanings:</b><br>
return_trailing_4q = total compounded return over t-3 to t

return_mean_4q = average quarterly return over t-3 to t

return_std_4q = variability of quarterly returns over t-3 to t

return_min_4q = worst quarterly return over t-3 to t

positive_quarter_share_4q = proportion of the last 4 quarters with positive returns

In [16]:
# inspect results
performance_features_4q = [
    "return_trailing_4q",
    "return_mean_4q",
    "return_std_4q",
    "return_min_4q",
    "positive_quarter_share_4q",
]

hist_model_mysuper_features[
    performance_features_4q
].describe()

,return_trailing_4q,return_mean_4q,return_std_4q,return_min_4q,positive_quarter_share_4q
count,12176.000000,12176.000000,12176.000000,12176.000000,12176.000000
mean,0.075215,0.018438,0.032825,-0.023338,0.755626
std,0.071414,0.016664,0.020246,0.039705,0.195507
min,-0.114451,-0.028650,0.000000,-0.177122,0.000000
25%,0.036310,0.010156,0.019993,-0.040755,0.750000
50%,0.082569,0.020413,0.027996,-0.010104,0.750000
75%,0.115614,0.027950,0.038938,0.001102,1.000000
max,0.372972,0.082475,0.131923,0.078200,1.000000


In [17]:
hist_model_mysuper_features[
    performance_features_4q
].isna().sum()

return_trailing_4q           1543
return_mean_4q               1543
return_std_4q                1543
return_min_4q                1543
positive_quarter_share_4q    1543
dtype: int64